In [1]:
"""
Modbus TCP (Python)
@author:    Eder H. N. Silva
Date:       April 15, 2025
"""
%reset

Once deleted, variables cannot be recovered. Proceed (y/[n])?  y


In [3]:
##IMPORTING LIBRARIES
import time
import struct
import numpy as np
from pyModbusTCP.client import ModbusClient

| **Data Type**                            | **Details**                                                                                                                                                                                                 |
|------------------------------------------|------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| **Bit**                                  | Bit information. The variable can receive binary values: **TRUE (1, ON)** or **FALSE (0, OFF)**.                                                                                                            |
| **Word [Signed]**                        | 16-bit signed integer. Accepts values from $-32,768$ to $+32,767$.                                                                                                                                      |
| **Word [Unsigned] / Bit string (16-bit)**| Unsigned 16-bit integer. Accepts values from $0$ to $65,535$.                                                                                                                                           |
| **Double Word [Signed]**                 | 32-bit signed integer. Accepts values from $-2,147,483,648$ to $+2,147,483,647$.                                                                                                                       |
| **Double Word [Unsigned] / Bit string (32-bit)** | Unsigned 32-bit integer. Accepts values from **0** to $4,294,967,296$.                                                                                                                           |
| **FLOAT [Single Precision]**             | 32-bit floating point number. Accepts values from $-2^{128}$ to $-2^{-126}$, $0$, and $2^{-126}$ to $2^{128}$.                                                                                   |
| **String**                               | Variable-length character data. Can store between $0$ and $255$ characters, defined like `String(20)` for a 20-character string.                                                                          |


## How to Read in Modbus TCP

In [19]:
## PLC CONNECTION
c = ModbusClient(host="192.168.15.250", port=502, auto_open=True, auto_close=True)

| **Variable**              | **Address** | **Size**    | **Type (Struct Format)** |
|---------------------------|-------------|-------------|--------------------------|
| Word Unsigned             | 0           | 16 bits     | `>H`                     |
| Word Signed               | 2           | 16 bits     | `>h`                     |
| Double Word Unsigned      | 4 and 5     | 32 bits     | `>I`                     |
| Double Word Signed        | 6 and 7     | 32 bits     | `>i`                     |
| Float (IEEE 754)         | 8 and 9     | 32 bits     | `>f`                     |

In [34]:
## READING 10 REGISTERS (0 TO 9)
registers = c.read_holding_registers(0, 10)
if registers:
    ## UNSIGNED WORD (ADDRESS 0)
    word_unsigned = struct.unpack('>H', struct.pack('>H', registers[0]))[0]

    ## WORD WITH SIGN (ADDRESS 2)
    word_signed = struct.unpack('>h', struct.pack('>H', registers[2]))[0]

    ## DOUBLE WORD WITHOUT SIGN (ADDRESS 4 AND 5)
    double_word_unsigned = struct.unpack('>I', struct.pack('>HH', registers[5], registers[4]))[0]
    
    ## SIGNED DOUBLE WORD (ADDRESS 6 AND 7)
    double_word_signed = struct.unpack('>i', struct.pack('>HH', registers[7], registers[6]))[0]

    ## FLOAT (ADDRESS 8 AND 9)
    float_value = struct.unpack('>f', struct.pack('>HH', registers[9], registers[8]))[0]

    print(f"[INFO] Word Unsigned : {word_unsigned}")
    print(f"[INFO] Word Signed   : {word_signed}")
    print(f"[INFO] Double Word Unsigned : {double_word_unsigned}")
    print(f"[INFO] Double Word Signed   : {double_word_signed}")
    print(f"[INFO] Float Value          : {float_value}")

else:
    print("[INFO] Error reading Modbus registers")

[INFO] Word Unsigned : 65535
[INFO] Word Signed   : -32768
[INFO] Double Word Unsigned : 1200000000
[INFO] Double Word Signed   : -2147483648
[INFO] Float Value          : 1.6180340051651


| **Value**                  | **Address**   | **Number of Words** | **How to Write**                              |
|----------------------------|----------------|---------------------|-----------------------------------------------|
| 65535 (Word Unsigned)      | 10             | 1 word (16 bits)    | Direct integer                               |
| -32768 (Word Signed)       | 12             | 1 word (16 bits)    | `value & 0xFFFF` to adjust signed value     |
| 4294967295 (DWord Unsigned)| 14 and 15      | 2 words (32 bits)   | Split into High Word and Low Word            |
| -2147483648 (DWord Signed) | 16 and 17      | 2 words (32 bits)   | Split into High Word and Low Word            |
| 3.14159265359 (Float)      | 18 and 19      | 2 words (32 bits)   | Use `struct.pack('>f', value)`              |

## How to Write in Modbus TCP

In [51]:
## DATA FOR WRITING
## WORD UNSIGNED -> 65535
word_unsigned = [65535]

## WORD SIGNED -> -32768 (CONVERTED TO UNSIGNED FORMAT FOR MODBUS)
word_signed = [(-32768) & 0xFFFF]

## DWORD UNSIGNED -> 1200000000
dword_unsigned = struct.unpack('<HH', struct.pack('>I', 1200000000))

## DWORD SIGNED -> -2147483648
dword_signed   = struct.unpack('<HH', struct.pack('>i', -2147483648))

## FLOAT -> 3.14159265359
float_value    = struct.unpack('<HH', struct.pack('>f', 3.14159265359))

## CONCATENATE ALL VALUES
## NOTE: WILL BE WRITTEN IN POSITIONS: 10 TO 19
data_to_write = word_unsigned + [0] + word_signed + [0] + list(dword_unsigned) + list(dword_signed) + list(float_value)

## WRITING TO MODBUS
ok = c.write_multiple_registers(10, data_to_write)

if ok:
    print("[INFO] Data Written Successfully!")
else:
    print("[INFO] Error Writing Data")

[INFO] Data Written Successfully!


| **Address** | **Value to be Written**                                  |
|-------------|-----------------------------------------------------------|
| 10          | 65535                                                    |
| 11          | 0                                                        |
| 12          | 32768 *(representation of -32768)*                       |
| 13          | 0                                                        |
| 14          | High Word of DWord Unsigned                               |
| 15          | Low Word of DWord Unsigned                                |
| 16          | High Word of DWord Signed                                 |
| 17          | Low Word of DWord Signed                                  |
| 18          | High Word of FLOAT                                        |
| 19          | Low Word of FLOAT                                         |

In [53]:
## DATA FOR WRITING
## WORD UNSIGNED -> 65535
word_unsigned = [65535]

## WORD SIGNED -> -32768 (CONVERTED TO UNSIGNED FORMAT FOR MODBUS)
word_signed = [(-32768) & 0xFFFF]

## DWORD UNSIGNED -> 4294967295
dword_unsigned = struct.unpack('>HH', struct.pack('>I', 1200000000))

## DWORD SIGNED -> -2147483648
dword_signed = struct.unpack('>HH', struct.pack('>i', -2147483648))

## FLOAT -> 3.14159265359
float_value = struct.unpack('>HH', struct.pack('>f', 1.61803398875))

## CONCATENATE ALL VALUES
## NOTE: WILL BE WRITTEN IN POSITIONS: 10 TO 19
data_to_write = word_unsigned + [0] + word_signed + [0] + list(dword_unsigned) + list(dword_signed) + list(float_value)
data_to_write 

[65535, 0, 32768, 0, 34375, 140, 128, 0, 16335, 7101]

In [ ]:
## DATA FOR WRITING
## WORD UNSIGNED -> 65535
word_unsigned = [65535]

## WORD SIGNED -> -32768 (CONVERTED TO UNSIGNED FORMAT FOR MODBUS)
word_signed = [(-32768) & 0xFFFF]

## DWORD UNSIGNED -> 4294967295
dword_unsigned = struct.unpack('<HH', struct.pack('>I', 1200000000))

## DWORD SIGNED -> -2147483648
dword_signed   = struct.unpack('<HH', struct.pack('>i', -2147483648))

## FLOAT -> 3.14159265359
float_value    = struct.unpack('<HH', struct.pack('>f', 1.61803398875))

## CONCATENATE ALL VALUES
## NOTE: WILL BE WRITTEN IN POSITIONS: 10 TO 19
data_to_write = word_unsigned + [0] + word_signed + [0] + list(dword_unsigned) + list(dword_signed) + list(float_value)
data_to_write 

## How to Write a String in Modbus TCP
Let's write the string "SMART" at address 10:

In [ ]:
## STRING I WANT TO SEND
my_string = "SMART"

## GUARANTEES EVEN NUMBER OF CHARACTERS
if len(my_string) % 2 != 0:
    my_string += '\x00'  # ADD FINAL NULL IF ODD

## CONVERT TO WORD
words = [struct.unpack('>H', my_string[i:i+2].encode())[0] for i in range(0, len(my_string), 2)]

## WRITE THE WORDS STARTING FROM ADDRESS 20
ok = c.write_multiple_registers(20, words)

if ok:
    print("[INFO] String written successfully!")
else:
    print("[INFO] Error writing string")